# 🔀 Notebook 3: Sharding and Partitioning

When one server can't handle the write load, distribute it across multiple servers. The key is choosing how to split the data.

## Learning Objectives

By the end of this notebook, you'll understand:
- Horizontal vs vertical partitioning
- Choosing effective partition keys
- Avoiding hot spots
- Consistent hashing basics

In [ ]:
import hashlib
import random
from collections import defaultdict
from typing import List, Dict

print("✅ Ready to learn about sharding!")

## 🔀 Horizontal Sharding

In [ ]:
print("🔀 Horizontal Sharding")
print("=" * 60)
print("""
Split ROWS across multiple databases based on a key.

BEFORE (Single DB):
─────────────────────────────────────────────────────────────
┌─────────────────────────────────────────┐
│            All Posts                    │
│  user_id=1, user_id=2, ... user_id=N   │
│         (Bottleneck!)                   │
└─────────────────────────────────────────┘

AFTER (Sharded by user_id):
─────────────────────────────────────────────────────────────
┌─────────────┐  ┌─────────────┐  ┌─────────────┐
│   Shard 0   │  │   Shard 1   │  │   Shard 2   │
│ user_id % 3 │  │ user_id % 3 │  │ user_id % 3 │
│    = 0      │  │    = 1      │  │    = 2      │
└─────────────┘  └─────────────┘  └─────────────┘

• Each shard handles 1/3 of the writes
• Linear scaling: 3 shards = 3x capacity
""")

In [ ]:
class SimpleShardedDB:
    def __init__(self, num_shards: int):
        self.num_shards = num_shards
        self.shards = {i: [] for i in range(num_shards)}
        self.write_counts = {i: 0 for i in range(num_shards)}
    
    def get_shard(self, key: int) -> int:
        return key % self.num_shards
    
    def write(self, user_id: int, data: dict):
        shard_id = self.get_shard(user_id)
        self.shards[shard_id].append({"user_id": user_id, **data})
        self.write_counts[shard_id] += 1
    
    def get_distribution(self) -> dict:
        total = sum(self.write_counts.values())
        return {
            shard_id: {
                "count": count,
                "percentage": (count / total * 100) if total > 0 else 0
            }
            for shard_id, count in self.write_counts.items()
        }

print("🔬 Simulating Writes with Uniform User IDs")
print("=" * 60)

db = SimpleShardedDB(num_shards=4)

for user_id in range(1, 10001):
    db.write(user_id, {"action": "post"})

print("\n📊 Write Distribution (uniform user IDs):")
for shard_id, stats in db.get_distribution().items():
    bar = "█" * int(stats["percentage"] / 2)
    print(f"   Shard {shard_id}: {stats['count']:>5} writes ({stats['percentage']:.1f}%) {bar}")

counts = [db.write_counts[s] for s in range(db.num_shards)]
imbalance = max(counts) / (sum(counts) / len(counts))

print(f"\n✅ Even distribution when keys are uniform!")
print(f"   Imbalance factor: {imbalance:.2f}x "
      f"(1.00 = perfect, {db.num_shards:.2f} = everything on one shard)")

# Sequential ids under modulo are exactly balanced. If this drifts, the
# baseline the rest of the notebook compares against is broken.
assert imbalance < 1.05, (
    f"sequential ids under modulo should be near-perfectly balanced, got "
    f"{imbalance:.2f}x from {counts}"
)


## ⚠️ The Hot Spot Problem

In [ ]:
print("⚠️ Bad Partition Key: Country")
print("=" * 60)

class CountryShardedDB:
    def __init__(self):
        self.shards = defaultdict(list)
        self.write_counts = defaultdict(int)
    
    def write(self, country: str, data: dict):
        self.shards[country].append(data)
        self.write_counts[country] += 1

db = CountryShardedDB()

countries = {
    "USA": 330,
    "China": 1400,
    "India": 1380,
    "Brazil": 210,
    "Russia": 140,
    "Japan": 125,
    "Germany": 83,
    "UK": 67,
    "New Zealand": 5,
    "Iceland": 0.3
}

for country, population in countries.items():
    writes = int(population * 10)
    for _ in range(writes):
        db.write(country, {"action": "post"})

total = sum(db.write_counts.values())
print("\n📊 Write Distribution by Country:")
for country in sorted(db.write_counts.keys(), key=lambda x: db.write_counts[x], reverse=True):
    count = db.write_counts[country]
    pct = count / total * 100
    bar = "█" * int(pct / 2)
    print(f"   {country:12}: {count:>6} writes ({pct:>5.1f}%) {bar}")

# Read the skew off the data instead of quoting a number from memory.
top2 = sorted(db.write_counts.items(), key=lambda kv: kv[1], reverse=True)[:2]
top2_pct = sum(c for _, c in top2) / total * 100
imbalance = max(db.write_counts.values()) / (total / len(db.write_counts))

print(f"\n❌ {top2[0][0]} and {top2[1][0]} take {top2_pct:.0f}% of all writes!")
print(f"   Imbalance factor: {imbalance:.1f}x -- the busiest shard carries "
      f"{imbalance:.1f}x what an evenly-loaded one would.")
print("   This creates HOT SPOTS - some shards overloaded!")
print(f"   Meanwhile {top2[-1][0]}'s shard... and Iceland's... sit idle.")

# The section only teaches something if the skew is actually severe.
assert top2_pct > 60, f"expected the top 2 countries to dominate, got {top2_pct:.0f}%"
assert imbalance > 3, (
    f"expected a badly hot-spotted distribution, got only {imbalance:.1f}x imbalance"
)


## 🎯 Choosing Good Partition Keys

In [ ]:
print("🎯 Characteristics of Good Partition Keys")
print("=" * 60)
print("""
GOOD PARTITION KEYS:
─────────────────────────────────────────────────────────────
✅ High cardinality (many unique values)
✅ Uniform distribution of writes
✅ Matches your access patterns
✅ Doesn't change often

Examples:
• user_id - Good for user-centric data
• order_id - Good for e-commerce
• device_id - Good for IoT

─────────────────────────────────────────────────────────────

BAD PARTITION KEYS:
─────────────────────────────────────────────────────────────
❌ Low cardinality (few values)
❌ Skewed distribution
❌ Time-based (all writes go to "current" shard)
❌ Frequently changing

Examples:
• country - Skewed (China >> Iceland)
• status - Low cardinality (pending/active/done)
• timestamp - All "now" writes go to same shard
• celebrity_id - One key gets millions of writes
""")

## 🔗 Why Sharded Systems Bend Over Backwards to Avoid Cross-Shard Writes

"Matches your access patterns" is the cheapest-sounding rule in the list above
and the most expensive one to get wrong. The moment a single logical operation
has to write two shards, a plain `BEGIN … COMMIT` no longer covers it: each
shard has its own WAL and reaches its own commit decision.

The textbook fix is **two-phase commit (2PC)**:

```
coordinator ──PREPARE──▶ shard A     A durably logs "prepared", HOLDS its locks
            ──PREPARE──▶ shard B     B durably logs "prepared", HOLDS its locks
            ◀──YES/YES──
            ──COMMIT───▶ A, B        second round of durable writes, locks released
```

What that actually costs:

| cost | why |
|---|---|
| ~2x latency | two round trips and two `fsync`s per participant, not one |
| locks held across the network | every participant holds row locks for the *whole* protocol, including the coordinator's think time |
| **blocking** on coordinator failure | if the coordinator dies after PREPARE, participants are stuck holding locks with no way to decide — they cannot unilaterally abort |
| throughput of the slowest shard | the operation cannot commit faster than its worst participant, so tail latency compounds |

So production systems mostly design the problem away rather than pay for it:

- **Co-locate.** Choose the key so everything one transaction touches lands on
  one shard — an order and its line items keyed by `order_id`, a user's whole
  subtree keyed by `user_id`. One shard means one ordinary local transaction.
- **Saga.** Split into per-shard local transactions plus compensating actions
  for rollback, accepting a visible window of inconsistency.
- **Move the invariant.** Keep the thing that needs a strict invariant (a
  ledger, a stock count) on a single shard and shard everything around it.

Rule of thumb: **if your partition key forces cross-shard transactions on the
hot path, it is the wrong partition key.** That is a schema problem, and 2PC is
not a fix for it — it is the interest payment.

In [ ]:
print("🎲 Hash-Based Sharding")
print("=" * 60)

def hash_shard(key: str, num_shards: int) -> int:
    hash_value = int(hashlib.md5(str(key).encode()).hexdigest(), 16)
    return hash_value % num_shards

db = SimpleShardedDB(num_shards=4)
# Swap the routing function: hash first, then modulo. Same interface, and the
# same writes as the country demo above -- only the key changes.
db.get_shard = lambda key: hash_shard(key, 4)

for country, population in countries.items():
    writes = int(population * 10)
    for i in range(writes):
        db.write(f"{country}_{i}", {"action": "post"})

total = sum(db.write_counts.values())
print("\n📊 Distribution with Hash-Based Sharding:")
for shard_id in sorted(db.write_counts.keys()):
    count = db.write_counts[shard_id]
    pct = count / total * 100
    bar = "█" * int(pct / 2)
    print(f"   Shard {shard_id}: {count:>6} writes ({pct:>5.1f}%) {bar}")

# Same writes, sharded on the raw country name, for a like-for-like number.
by_country = {c: int(p * 10) for c, p in countries.items()}
country_imbalance = max(by_country.values()) / (sum(by_country.values()) / len(by_country))

hash_imbalance = max(db.write_counts.values()) / (total / db.num_shards)
print(f"\n✅ Hash spreads writes evenly regardless of input distribution!")
print(f"   Imbalance factor: {hash_imbalance:.2f}x "
      f"(was {country_imbalance:.1f}x when we sharded on the raw country name).")
print("   What you gave up: locality. All of China's rows are now scattered")
print("   across all 4 shards, so 'everything from China' is a fan-out query.")

# Same input, same skew, different key -- the hot spot has to be gone.
assert hash_imbalance < 1.05, (
    f"md5 of a high-cardinality key should balance to within 5%, got "
    f"{hash_imbalance:.2f}x from {dict(db.write_counts)}"
)
assert hash_imbalance < country_imbalance / 3, (
    f"hashing must dramatically beat the raw country key, got "
    f"{hash_imbalance:.2f}x vs {country_imbalance:.2f}x"
)


## ⏱️ The Bad Key That Hides: Anything That Only Goes Up

Skew from a *value* distribution (China vs Iceland) is easy to spot — you can
see it in a `GROUP BY`. The subtler killer is a **monotonically increasing
key**: `created_at`, a `SERIAL` id, a UUIDv1/v7 timestamp prefix. Its
distribution is perfectly uniform. It still hot-spots, because of *when* the
values appear rather than *how many* there are.

It only bites when combined with **range** sharding, where each shard owns a
contiguous interval of the key space. Range sharding is great for range scans…
and catastrophic for writes, because "now" is always inside exactly one
interval. Every new row lands on the newest shard while the rest idle. This is
the HBase / DynamoDB "sequential key" anti-pattern, and it is the reason the
quiz below calls `timestamp` a bad partition key.

Below we shard the same 10,000 timestamp-ordered events three ways and measure
the **imbalance factor** = writes on the hottest shard ÷ writes on an average
shard. With 4 shards, `1.0` is perfect and `4.0` means one shard is doing all
the work.

In [ ]:
print("⏱️ Monotonically increasing key + range sharding = one hot shard")
print("=" * 60)

NUM_SHARDS = 4
NUM_EVENTS = 10_000

# Event ids arrive in order: 1, 2, 3, ... exactly like a SERIAL column or a
# timestamp. That the key never goes backwards is the whole point.
event_ids = list(range(1, NUM_EVENTS + 1))


def imbalance_factor(counts: dict) -> float:
    """Hottest shard's load / average shard's load.

    1.0 = perfectly even. NUM_SHARDS = one shard is taking everything.
    """
    return max(counts.values()) / (sum(counts.values()) / NUM_SHARDS)


# --- Strategy A: RANGE sharding on the monotonic key ----------------------
# Shard i owns ids [i * range_size, (i+1) * range_size). The boundaries are
# sized for the ids that exist *today*.
range_size = NUM_EVENTS // NUM_SHARDS

def range_shard(eid: int) -> int:
    return min((eid - 1) // range_size, NUM_SHARDS - 1)

range_backfill = {s: 0 for s in range(NUM_SHARDS)}
for eid in event_ids:
    range_backfill[range_shard(eid)] += 1

# Now the next hour of production traffic: 10,000 *new* events, ids continuing
# upward. Every one of them falls past the last boundary.
range_live = {s: 0 for s in range(NUM_SHARDS)}
for eid in range(NUM_EVENTS + 1, NUM_EVENTS * 2 + 1):
    range_live[range_shard(eid)] += 1

# --- Strategy B: MODULO on the monotonic key ------------------------------
mod_counts = {s: 0 for s in range(NUM_SHARDS)}
for eid in event_ids:
    mod_counts[eid % NUM_SHARDS] += 1

# --- Strategy C: HASH of a high-cardinality, non-monotonic key ------------
# 5,000 users generating 10,000 events. The key is who did it, not when.
hash_counts = {s: 0 for s in range(NUM_SHARDS)}
for eid in event_ids:
    hash_counts[hash_shard(f"user_{eid % 5000}", NUM_SHARDS)] += 1

print("\n📊 Imbalance factor (1.00 = perfect, 4.00 = one shard does everything):")
print(f"   A. RANGE on monotonic id, historical backfill : "
      f"{imbalance_factor(range_backfill):.2f}x")
print(f"   A. RANGE on monotonic id, NEXT HOUR of traffic: "
      f"{imbalance_factor(range_live):.2f}x  🔥")
print(f"   B. MODULO on monotonic id                     : "
      f"{imbalance_factor(mod_counts):.2f}x")
print(f"   C. HASH of user_id (not monotonic)            : "
      f"{imbalance_factor(hash_counts):.2f}x")

for label, counts in [("A. RANGE, next hour", range_live),
                      ("B. MODULO", mod_counts),
                      ("C. HASH of user_id", hash_counts)]:
    shard_total = sum(counts.values())
    print(f"\n   {label}:")
    for s in range(NUM_SHARDS):
        pct = counts[s] / shard_total * 100
        print(f"      Shard {s}: {counts[s]:>6} ({pct:>5.1f}%) {'█' * int(pct / 2)}")

print("""
💡 Two lessons are buried in those four numbers:

   1. RANGE sharding on a monotonic key looks PERFECT on a historical backfill
      (1.00x -- the old ids really are spread evenly over the ranges you sized
      for them) and is 4.00x broken the instant new traffic arrives. Load-
      testing against a backfill is exactly how this bug reaches production.

   2. MODULO on a monotonic key is perfectly balanced for writes -- and it
      destroys locality. A "last 10 minutes" query now has to hit every shard
      and merge. Hashing a non-monotonic key (C) buys you balance AND keeps one
      entity's rows together, which is why user_id beats timestamp.

   Range sharding is not wrong -- it is what you want for time-series *reads*.
   The fix is to range-shard on time but prefix the key with something that
   spreads (user_id, a hash bucket) so "now" is not a single interval.
""")

# The backfill must look fine and the live traffic must be maximally skewed --
# that contrast IS the lesson. If either side moves, the section stops working.
assert imbalance_factor(range_backfill) < 1.05, (
    f"a historical backfill should look balanced under range sharding, got "
    f"{imbalance_factor(range_backfill):.2f}x"
)
assert imbalance_factor(range_live) == NUM_SHARDS, (
    f"every new monotonic id must land on the last shard, got "
    f"{imbalance_factor(range_live):.2f}x from {range_live}"
)
assert imbalance_factor(mod_counts) < 1.05, (
    f"modulo on sequential ids is exactly even, got {imbalance_factor(mod_counts):.2f}x"
)
assert imbalance_factor(hash_counts) < 1.10, (
    f"hashing a high-cardinality key should balance to ~10%, got "
    f"{imbalance_factor(hash_counts):.2f}x from {hash_counts}"
)


## 🧭 Modulo Sharding Breaks When You Add a Node

`user_id % N` is simple, but **changing `N` remaps almost every key**.
If you grow from 4 → 5 shards, roughly 4 out of every 5 keys move. In
practice that means a multi-hour rebalancing storm.

```
N=4:  key 42 -> shard 42 % 4 = 2
N=5:  key 42 -> shard 42 % 5 = 2   ✅ same shard (lucky!)
N=4:  key 17 -> shard 17 % 4 = 1
N=5:  key 17 -> shard 17 % 5 = 2   ❌ moved
```

**Consistent hashing** is the classic fix: when you add or remove a node,
only `K/N` keys move instead of almost all of them.


## ⭕ Consistent Hashing (The Intuition)

Imagine all possible hashes laid out on a **circle** (0 -> 2^32 -> 0 again).
Every shard gets placed at several positions on that circle (these are
called **virtual nodes**, or *vnodes*).

To find which shard owns a key:

1. Hash the key -> a point on the circle.
2. Walk **clockwise** until you hit a shard's vnode.
3. That shard owns the key.

Adding a new shard only "steals" the slice of circle next to its vnodes —
most keys stay where they are.

This is how Cassandra, DynamoDB, and many CDN load balancers route data.


In [ ]:
from bisect import bisect

class ConsistentHashRing:
    """Minimal consistent-hash ring with virtual nodes.

    Each real shard owns vnodes_per_shard points on the ring. More vnodes
    -> smoother load distribution. Typical values: 100-500 per shard.
    """

    def __init__(self, shards, vnodes_per_shard: int = 100):
        self.vnodes_per_shard = vnodes_per_shard
        self.ring = {}
        self.sorted_points = []
        for s in shards:
            self.add_shard(s)

    def _hash(self, s: str) -> int:
        return int(hashlib.md5(s.encode()).hexdigest(), 16)

    def add_shard(self, shard_id):
        for v in range(self.vnodes_per_shard):
            point = self._hash(f"{shard_id}#{v}")
            self.ring[point] = shard_id
        self.sorted_points = sorted(self.ring.keys())

    def remove_shard(self, shard_id):
        for v in range(self.vnodes_per_shard):
            point = self._hash(f"{shard_id}#{v}")
            self.ring.pop(point, None)
        self.sorted_points = sorted(self.ring.keys())

    def get_shard(self, key) -> str:
        point = self._hash(str(key))
        i = bisect(self.sorted_points, point) % len(self.sorted_points)
        return self.ring[self.sorted_points[i]]


print("⭕ Consistent Hashing: adding a shard with minimal movement")
print("=" * 60)

ring4 = ConsistentHashRing(["A", "B", "C", "D"], vnodes_per_shard=200)
ring5 = ConsistentHashRing(["A", "B", "C", "D", "E"], vnodes_per_shard=200)

keys = [f"user_{i}" for i in range(10_000)]

dist4 = defaultdict(int)
for k in keys:
    dist4[ring4.get_shard(k)] += 1

print("\n📊 With 4 shards (A-D):")
for s, n in sorted(dist4.items()):
    bar = "#" * (n // 100)
    print(f"   Shard {s}: {n:>5} ({n/len(keys)*100:.1f}%) {bar}")

moved = sum(1 for k in keys if ring4.get_shard(k) != ring5.get_shard(k))
modulo_moved = sum(1 for i in range(len(keys)) if (i % 4) != (i % 5))

print(f"\n🔁 Keys that move when going 4 -> 5 shards:")
print(f"   Consistent hash: {moved:>5} / {len(keys)} ({moved/len(keys)*100:.1f}%)")
print(f"   Plain modulo:    {modulo_moved:>5} / {len(keys)} ({modulo_moved/len(keys)*100:.1f}%)")
print("\n✅ Consistent hashing ~ K/N keys move; modulo moves almost everything.")

# Consistent hashing's entire selling point is the movement ratio. Pin it.
ch_frac = moved / len(keys)
mod_frac = modulo_moved / len(keys)
assert ch_frac < 0.30, (
    f"adding the 5th shard should move ~1/5 of keys, consistent hashing moved "
    f"{ch_frac * 100:.1f}%"
)
assert mod_frac > 0.70, (
    f"plain modulo should move ~4/5 of keys on 4->5, moved {mod_frac * 100:.1f}%"
)
assert moved < modulo_moved / 2, (
    f"consistent hashing must move far fewer keys than modulo, got "
    f"{moved} vs {modulo_moved}"
)
print(f"\n💰 What that ratio costs in practice: rebalancing means streaming rows")
print(f"   over the network while both nodes take live traffic. Moving "
      f"{mod_frac * 100:.0f}% of a")
print(f"   1 TB dataset instead of {ch_frac * 100:.0f}% is the difference between an "
      f"afternoon and a week.")


## 🗄️ Real-World: PostgreSQL Declarative Partitioning

Everything we've shown so far has been *in-memory* simulations. But PostgreSQL
has **built-in sharding inside a single database** called **declarative
partitioning**. It doesn't give you multi-server scaling, but it:

- Splits one big logical table into many smaller **partitions** (physical tables).
- Lets the planner scan only the relevant partition (**partition pruning**).
- Makes it trivial to **drop old data** (just `DROP` an old partition).

This is exactly how **TimescaleDB** builds time-series superpowers on top of
PostgreSQL, and how teams routinely store billions of rows in a single
database.

There are two common flavors:

```
HASH partitioning            RANGE partitioning
(shard by user_id)           (shard by time)

events_p0  (user_id % 4 = 0)  events_2024_01
events_p1  (user_id % 4 = 1)  events_2024_02
events_p2  (user_id % 4 = 2)  events_2024_03
events_p3  (user_id % 4 = 3)  events_2024_04
```

Below we build a HASH-partitioned events table, insert rows, and then ask
PostgreSQL to show us how evenly the writes landed.


In [ ]:
import psycopg2

DB_CONFIG = {"host": "localhost", "port": 5432, "database": "writes_demo",
             "user": "demo", "password": "demo"}

conn = psycopg2.connect(**DB_CONFIG)
conn.autocommit = True
cur = conn.cursor()

# Start fresh so we can re-run this cell.
cur.execute("DROP TABLE IF EXISTS events_partitioned CASCADE")

# 1. Declare the parent table as PARTITION BY HASH on user_id.
#    The parent has no storage of its own; rows live in child partitions.
cur.execute("""
    CREATE TABLE events_partitioned (
        id         BIGSERIAL,
        user_id    INTEGER NOT NULL,
        event_type VARCHAR(50),
        payload    JSONB,
        created_at TIMESTAMP DEFAULT now(),
        PRIMARY KEY (id, user_id)
    ) PARTITION BY HASH (user_id);
""")

# 2. Create 4 child partitions. Each one owns rows whose hash(user_id) mod 4
#    equals its remainder. PostgreSQL routes INSERTs automatically.
for i in range(4):
    cur.execute(f"""
        CREATE TABLE events_p{i}
        PARTITION OF events_partitioned
        FOR VALUES WITH (MODULUS 4, REMAINDER {i});
    """)

# 3. Insert 10,000 rows with uniformly-distributed user_ids.
#    setseed() makes PostgreSQL's random() reproducible for this session, so
#    the row counts below are the same every run instead of drifting.
cur.execute("SELECT setseed(0.42)")
cur.execute("""
    INSERT INTO events_partitioned (user_id, event_type, payload)
    SELECT (random() * 100000)::int, 'click', '{"x": 1}'::jsonb
    FROM generate_series(1, 10000);
""")

# 4. Ask each partition how many rows it holds - this is the "shard balance".
print("📊 Rows per partition (HASH on user_id):")
partition_counts = []
for i in range(4):
    cur.execute(f"SELECT count(*) FROM events_p{i}")
    n = cur.fetchone()[0]
    partition_counts.append(n)
    bar = "#" * (n // 100)
    print(f"   events_p{i}: {n:>5} rows {bar}")

imbalance = max(partition_counts) / (sum(partition_counts) / 4)
print(f"\n   Total: {sum(partition_counts):,} rows, imbalance factor {imbalance:.2f}x")

assert sum(partition_counts) == 10_000, (
    f"rows went missing during routing: {partition_counts}"
)
assert imbalance < 1.10, (
    f"HASH partitioning should land within ~10% of even, got {partition_counts}"
)

# 5. EXPLAIN shows that a query filtered by user_id hits only ONE partition.
#    This is "partition pruning" - the planner skips 3 of the 4 tables.
cur.execute("""
    EXPLAIN (COSTS OFF)
    SELECT count(*) FROM events_partitioned WHERE user_id = 42;
""")
plan = [row[0] for row in cur.fetchall()]
print("\n🔍 Partition pruning (note the single 'Scan on events_pX'):")
for line in plan:
    print("   ", line)

scanned = sorted(
    p for p in (f"events_p{i}" for i in range(4))
    if any(p in line for line in plan)
)
print(f"\n   Partitions the planner will actually touch: {scanned}")

# Pruning is the entire point of the section. If the planner starts scanning
# all four partitions, the demo is claiming something it no longer shows.
assert len(scanned) == 1, (
    f"expected partition pruning down to exactly 1 partition, planner kept "
    f"{scanned}"
)

cur.close()
conn.close()


## 📐 Vertical Partitioning

In [ ]:
print("📐 Vertical Partitioning")
print("=" * 60)
print("""
Split COLUMNS into different tables/databases based on access patterns.

BEFORE (Monolithic):
─────────────────────────────────────────────────────────────
┌─────────────────────────────────────────────────────────┐
│                        posts                             │
├──────┬─────────┬───────────┬────────────┬───────────────┤
│  id  │ content │ like_cnt  │ view_cnt   │ share_cnt     │
│      │ (write  │ (frequent │ (very      │ (occasional   │
│      │  once)  │  updates) │  frequent) │  updates)     │
└──────┴─────────┴───────────┴────────────┴───────────────┘

Problem: Like/view updates cause locks on content reads!

AFTER (Vertically Partitioned):
─────────────────────────────────────────────────────────────
┌─────────────────┐    ┌──────────────────────────────────┐
│  post_content   │    │         post_metrics             │
├──────┬──────────┤    ├──────┬─────────┬────────┬────────┤
│  id  │ content  │    │  id  │like_cnt │view_cnt│share_ct│
│      │          │    │      │         │        │        │
│ (write once,    │    │ (high-frequency counter updates) │
│  read often)    │    │                                  │
└─────────────────┘    └──────────────────────────────────┘

• Different write patterns = different optimizations
• Content: Optimized for reads (more indexes)
• Metrics: Optimized for writes (minimal indexes)
""")

## 🧪 Quick Quiz

1. **Why is user_id usually a good partition key?**

2. **What's the problem with using timestamp as a partition key?**

3. **When would you use vertical vs horizontal partitioning?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Why user_id is good:")
print("   - High cardinality (millions of users)")
print("   - Users spread writes naturally")
print("   - Matches common access patterns")
print("   - User's data often accessed together")
print()
print("2. Problem with timestamp:")
print("   - All 'current' writes go to same shard")
print("   - Creates sequential hot spot")
print("   - Only latest shard is ever busy")
print()
print("3. Vertical vs Horizontal:")
print("   Vertical: Different access patterns per column")
print("            (content vs counters)")
print("   Horizontal: Same schema, too much data")
print("              (split rows across shards)")

## 📚 Summary

### Key Takeaways

1. **Horizontal sharding** - Split rows across servers
2. **Choose keys wisely** - High cardinality, uniform distribution
3. **Hash for uniformity** - Spreads skewed keys evenly
4. **Vertical partitioning** - Separate by access pattern
5. **Avoid hot spots** - They defeat the purpose of sharding

### Next Up

In **Notebook 4**, we'll learn about queues and load shedding:
- Handling bursty traffic
- Async write patterns
- Graceful degradation